# H03：14:30 Feature / T+1 Label 验证

本 Notebook 恢复历史验证入口，复用 2026-09-08 的验证脚本，对当前固定源码执行
技术复核。运行事实保存在输出；研究状态、结论和采用决定由 [README 的 H03](README.md#h03) 拥有。
比较对象是数据构建和对象生命周期，不评价因子预测价值。

真实数据验证固定四组 T/T+1，另保留一个缺失输入负例。精确相等和失败边界是主要
验证问题，使用结果表呈现，不生成预测效果图。

**2026-09-13 保存运行事实**：7 个代码单元从头执行成功，53 项定向回归通过；四组样本共 20,669 行，八个 Feature/Label payload 与参考版本精确一致。16 个对象复用后身份不变，1 个缺失输入负例符合预期。

## 方法与复跑

当前源码固定为 `9e5f9bb5472cee6262e32347e4cf6ffb715dcd25`；参考源码为历史
`cd88f0c` 归档。使用相同 30 个 calendar/minute/factor 对象和锁定项目环境。
`validate_alignment.py` 沿用原有真实 CLI、schema、key、值、null、复用和失败断言；
本轮只固定 CLI 输出宽度 `COLUMNS=240`，避免长路径换行拆开负例日期。
现有 pytest 负责手算、时间边界、稀疏窗口、大整数、rank 和无输入读取等回归。

### 输入与解释边界

- Feature 只使用 T 日 14:30 前 CONTINUOUS 分钟；Label 使用 T/T+1 的 `[14:31,14:36)`。
- 归档是历史隔离输入：payload 内容固定，Meta 沿用当时的隔离表示；不审计正式 raw lineage。
- 不查询现在的正式数据来填补负例，也不把历史后续补齐改变为本轮输入。
- 日期、输入、参考版本和断言沿用 README 预注册；任何必要断言失败即停止并保存现场。

在项目锁定环境中选择 Python 3.13.13 kernel，按顺序运行全部单元。
首次准备项目依赖可用 `uv sync --locked --extra test`。证据目录移动后，修改下方
`EVIDENCE` 或通过 `STOCK1430_EVIDENCE_ROOT` 指定解包目录。每次执行新建隔离 run，
只使用无效占位凭证。环境、原始命令、stdout/stderr 和失败结果写入该 run。
源码与输入/参考归档都随证据保留，不依赖历史 `/tmp` 目录。

In [1]:
import hashlib
import json
import os
import subprocess
import sys
import tempfile
import xml.etree.ElementTree as ET
from collections.abc import Sequence
from importlib import metadata
from pathlib import Path

import pandas as pd
from IPython.display import display

EVIDENCE = Path(
    os.environ.get(
        "STOCK1430_EVIDENCE_ROOT",
        "/home/wsw/app/research-evidence/stock-1430-evidence-2026-09-13-a74ry7d6",
    )
)
SOURCE = EVIDENCE / "source"
LOCKED_PYTHON = Path(sys.executable)
BASELINE = "9e5f9bb5472cee6262e32347e4cf6ffb715dcd25"
PAIRS = (
    ("2025-11-18", "2025-11-19"),
    ("2025-12-31", "2026-01-05"),
    ("2026-04-30", "2026-05-06"),
    ("2026-07-27", "2026-07-28"),
)
assert sys.version_info[:3] == (3, 13, 13)
versions = {
    name: metadata.version(name) for name in ("numpy", "pandas", "pyarrow", "pytest")
}
assert versions == {
    "numpy": "2.5.1",
    "pandas": "3.0.2",
    "pyarrow": "25.0.0",
    "pytest": "9.0.3",
}
RUN = Path(tempfile.mkdtemp(prefix="h03-run-", dir=EVIDENCE))
environment = {
    "PATH": str(LOCKED_PYTHON.parent) + ":/usr/bin:/bin",
    "LANG": "C.UTF-8",
    "ENV": "dev",
    "ZERO_STORAGE_ROOT": str(RUN / "unused-store"),
    "PYTHONHASHSEED": "0",
    "PYTHONDONTWRITEBYTECODE": "1",
    "PYTHONPATH": str(SOURCE),
}
context = {
    "commit": BASELINE,
    "python": sys.version,
    "executable": str(LOCKED_PYTHON),
    "packages": dict(
        sorted(
            (package.metadata["Name"], package.version)
            for package in metadata.distributions()
        )
    ),
    "pairs": PAIRS,
    "negative_pair": ["2025-11-24", "2025-11-25"],
    "randomness": "no randomized computation; PYTHONHASHSEED=0",
}
(RUN / "context.json").write_text(json.dumps(context, indent=2) + "\n")
print("隔离运行目录：", RUN)
display(pd.DataFrame(PAIRS, columns=["T", "T+1"]))

隔离运行目录： /home/wsw/app/research-evidence/stock-1430-evidence-2026-09-13-a74ry7d6/h03-run-sl1ucp7y


,T,T+1
0,2025-11-18,2025-11-19
1,2025-12-31,2026-01-05
2,2026-04-30,2026-05-06
3,2026-07-27,2026-07-28


## 1. 核对可恢复源码、验证脚本与输入

先检查 manifest 本身，再逐文件检查内容。输入 archive 的哈希绑定全部 payload 与
隔离 Meta；验证器随后按原始 30 对象 manifest 再核对实际提取输入。
可执行源码从固定 commit 快照运行，当前工作树后续修改不会静默改变本次基线。

In [2]:
def _sha256(path: Path) -> str:
    with path.open("rb") as reader:
        return hashlib.file_digest(reader, "sha256").hexdigest()


assert (
    _sha256(EVIDENCE / "source-manifest.json")
    == "e3b4d15371268cb493174ad6b05c8e9a438d94f96c1caf8858c0cb9bc6cf1404"
)
assert (
    _sha256(EVIDENCE / "artifact-manifest.json")
    == "d738a7b1d6267f7d94ad5abf42fe677d584cc05955971a4c5136749f85a2bf61"
)
source_manifest = json.loads((EVIDENCE / "source-manifest.json").read_text())
artifacts = json.loads((EVIDENCE / "artifact-manifest.json").read_text())
for item in source_manifest:
    assert _sha256(SOURCE / item["path"]) == item["sha256"], item["path"]
for item in artifacts:
    assert _sha256(EVIDENCE / item["path"]) == item["sha256"], item["path"]
assert (
    _sha256(SOURCE / "uv.lock")
    == "96125da32034e999352f8a0326f6bddb0c5a9408c8880a9197f2faebf5d4f512"
)
print(f"源码快照 {len(source_manifest)} 个文件及 3 个归档/脚本摘要一致。")
display(
    pd.DataFrame(artifacts)[["path", "size_bytes"]].rename(
        columns={"path": "证据文件", "size_bytes": "字节数"}
    )
)

源码快照 288 个文件及 3 个归档/脚本摘要一致。


,证据文件,字节数
0,input-archive.tar.gz,446067119
1,reference-source.tar,2109440
2,validate_alignment.py,11260


## 2. 手算、时间边界与发布回归

执行现有 H03 builder、Step、Access、CLI、request 和 workflow 测试。
覆盖四种窗口、14:30 及窗口外数据不影响对应产物、VWAP/factor、null/tie、
大整数、Feature/Label key、Meta 免读输入和部分成功续建。
测试日志与 JUnit 明细保留；本轮未修改运行代码，不重跑无关全仓测试。

In [3]:
def _execute_logged(argv: Sequence[str], *, name: str) -> None:
    with (RUN / f"{name}.log").open("w") as log:
        completed = subprocess.run(
            list(argv),
            cwd=SOURCE,
            env=environment,
            stdout=log,
            stderr=subprocess.STDOUT,
            check=False,
        )
    receipt = {
        "argv": list(argv),
        "cwd": str(SOURCE),
        "environment": environment,
        "exit_code": completed.returncode,
    }
    (RUN / f"{name}.command.json").write_text(json.dumps(receipt, indent=2) + "\n")
    assert completed.returncode == 0, f"执行失败，保留日志：{RUN / (name + '.log')}"


_execute_logged(
    [
        str(LOCKED_PYTHON),
        "-B",
        "-m",
        "pytest",
        "-q",
        "-p",
        "no:cacheprovider",
        "--junitxml",
        str(RUN / "regression.xml"),
        "tests/data_system/builders/test_stock_1430.py",
        "tests/data_system/steps/test_stock_1430_build.py",
        "tests/access/test_access.py",
        "tests/test_cli.py",
        "tests/jobs/test_requests.py",
        "tests/workflows/test_offline_daily_data.py",
        "-k",
        "stock_1430 and not fusion",
    ],
    name="regression",
)
suites = ET.parse(RUN / "regression.xml").getroot().findall("testsuite")
regression = {
    field: sum(int(suite.attrib[field]) for suite in suites)
    for field in ("tests", "failures", "errors", "skipped")
}
assert regression["tests"] > 0
assert regression["failures"] == regression["errors"] == regression["skipped"] == 0
print(f"H03 定向回归：{regression['tests']} passed；无失败、错误或跳过。")

H03 定向回归：53 passed；无失败、错误或跳过。


## 3. 两版真实 CLI 构建与复用

使用 `validate_alignment.py`；原脚本另存为 `validate_alignment-original.py`。
它在新的隔离存储中分别构建参考版本和
当前版本，并比较八个 Feature/Label payload；再由当前版本复用两版共 16 个对象。
`2025-11-24 → 2025-11-25` 负例必须退出 1，保留 Feature 且不发布 Label。
逐次 CLI 的参数、环境、输出和资源观测保存在 `validation/commands.json` 及同目录日志。

In [4]:
VALIDATION = RUN / "validation"
_execute_logged(
    [
        str(LOCKED_PYTHON),
        "-B",
        str(EVIDENCE / "validate_alignment.py"),
        "--candidate-source",
        str(SOURCE),
        "--reference-archive",
        str(EVIDENCE / "reference-source.tar"),
        "--input-archive",
        str(EVIDENCE / "input-archive.tar.gz"),
        "--output-directory",
        str(VALIDATION),
    ],
    name="real-validation",
)
summary = json.loads((VALIDATION / "summary.json").read_text())
comparison = json.loads((VALIDATION / "comparison.json").read_text())
commands = json.loads((VALIDATION / "commands.json").read_text())
assert summary["input_objects"] == 30 and summary["pairs_compared"] == 4
assert summary["payloads_compared"] == 8
assert summary["outputs_reused_without_change"] == 32
assert summary["negative_case_passed"]
assert (
    summary["input_manifest_file_sha256"]
    == "cfe55cf061bb6b42cecf8c6e0520d9680ec90aab0ece38b1b6405a7dd14b54e7"
)
assert (
    len(commands) == 17 and sum(command["exit_code"] == 1 for command in commands) == 1
)
print("8 次新建 CLI、8 次复用 CLI、1 次预设失败：验证器全部断言通过。")

8 次新建 CLI、8 次复用 CLI、1 次预设失败：验证器全部断言通过。


## 4. 四组样本结果

每行对应当前版本的一次首次 CLI。有效 Label 和缺失率均以当日完整 Feature
universe 为分母；Label 缺失不删除 Feature 行。耗时和 peak RSS 包含 CLI 进程开销，
只作观测，没有性能或容量通过门槛。

In [5]:
rows = []
for (trade_date, next_trade_date), result in zip(PAIRS, comparison, strict=True):
    assert result["trade_date"] == trade_date
    command = next(
        item
        for item in commands
        if item["stage"] == "candidate" and item["trade_date"] == trade_date
    )
    wall_seconds, peak_rss_kib = map(float, command["resource_observation"].split())
    rows.append(
        {
            "T": trade_date,
            "T+1": next_trade_date,
            "Feature/Label 行数": result["rows"],
            "有效 Label": result["valid_labels"],
            "Label 缺失率 %": 100 * (1 - result["valid_labels"] / result["rows"]),
            "CLI wall 秒": wall_seconds,
            "peak RSS MiB": peak_rss_kib / 1024,
        }
    )
overview = pd.DataFrame(rows)
display(overview.round({"Label 缺失率 %": 3, "CLI wall 秒": 3, "peak RSS MiB": 2}))
print("35/4 列 schema、三 key、值、null 和行序：四组与参考版本精确一致。")

,T,T+1,Feature/Label 行数,有效 Label,Label 缺失率 %,CLI wall 秒,peak RSS MiB
0,2025-11-18,2025-11-19,5157,5149,0.155,2.73,1086.96
1,2025-12-31,2026-01-05,5170,5158,0.232,2.69,1125.90
2,2026-04-30,2026-05-06,5150,5136,0.272,2.75,1085.14
3,2026-07-27,2026-07-28,5192,5188,0.077,2.73,1072.73


35/4 列 schema、三 key、值、null 和行序：四组与参考版本精确一致。


## 5. 32 列 Feature 有效值覆盖率

百分比分母为各日完整 Feature universe。列名去掉共同的 `f_l2_` 前缀和 `_rank`
标记，仅为缩短表格；最后的分钟数仍表示固定计划窗口。输出没有因缺失删行或填零。
`observed_minute_ratio` 的 100% 表示该列均有值，不表示计划分钟均被观察到。
本表只描述这四组冻结样本，不表示全历史质量或因子有效性。

In [6]:
coverage_columns = {}
for result in comparison:
    coverage_columns[result["trade_date"]] = {
        name.removeprefix("f_l2_").replace("_rank", ""): 100
        * (1 - null_count / result["rows"])
        for name, null_count in result["feature_nulls"].items()
    }
coverage = pd.DataFrame(coverage_columns)
metric_order = (
    "edge_vwap_return",
    "high_low_range",
    "notional",
    "trade_count",
    "average_trade_notional",
    "tick_signed_volume_ratio",
    "tick_signed_notional_ratio",
    "observed_minute_ratio",
)
coverage = coverage.reindex(
    [f"{metric}_{window}m" for window in (5, 15, 30, 60) for metric in metric_order]
)
assert coverage.shape == (32, 4) and coverage.notna().all().all()
coverage.index.name = "Feature / 计划窗口"
with pd.option_context("display.max_rows", 40, "display.max_columns", 8):
    display(coverage.round(3))

,2025-11-18,2025-12-31,2026-04-30,2026-07-27
Feature / 计划窗口,,,,
edge_vwap_return_5m,97.692,97.311,98.252,96.957
high_low_range_5m,99.961,99.923,99.903,99.981
notional_5m,99.961,99.923,99.903,99.981
trade_count_5m,99.961,99.923,99.903,99.981
average_trade_notional_5m,99.961,99.923,99.903,99.981
tick_signed_volume_ratio_5m,99.961,99.923,99.903,99.981
tick_signed_notional_ratio_5m,99.961,99.923,99.903,99.981
observed_minute_ratio_5m,100.000,100.000,100.000,100.000
edge_vwap_return_15m,98.061,97.079,97.981,97.111


## 6. 复用、缺失输入与证据保存

复用比较覆盖 SHA-256、字节数、mtime 和 inode。真实 CLI 复用及 pytest 的免读输入
场景共同约束对象生命周期；数据构建结果一致不能单独证明未读取上游。
30 个输入对象的三个隔离副本都保留前后指纹；本轮不访问正式存储。

In [7]:
output_before = json.loads((VALIDATION / "outputs-before-reuse.json").read_text())
output_after = json.loads((VALIDATION / "outputs-after-reuse.json").read_text())
inputs_before = json.loads((VALIDATION / "inputs-before.json").read_text())
inputs_after = json.loads((VALIDATION / "inputs-after.json").read_text())
assert output_before == output_after and len(output_after) == 32
assert inputs_before == inputs_after and len(inputs_after) == 180
for item in source_manifest:
    assert _sha256(SOURCE / item["path"]) == item["sha256"], item["path"]
checks = pd.DataFrame(
    [
        {"检查": "两版八个 payload 的逻辑内容", "数量": 8, "结果": "精确相等"},
        {"检查": "复用后 payload/Meta 文件身份", "数量": 32, "结果": "保持不变"},
        {"检查": "三份隔离输入文件身份", "数量": 180, "结果": "保持不变"},
        {
            "检查": "缺少 T+1 分钟的 CLI 负例",
            "数量": 1,
            "结果": "退出 1；Feature 保留，Label 未发布",
        },
    ]
)
display(checks)
result = {
    "commit": BASELINE,
    "regression": regression,
    "validation": summary,
    "overview": rows,
    "commands": commands,
    "assertions_passed": True,
    "source_manifest_sha256": _sha256(EVIDENCE / "source-manifest.json"),
    "artifact_manifest_sha256": _sha256(EVIDENCE / "artifact-manifest.json"),
}
(RUN / "result.json").write_text(json.dumps(result, indent=2) + "\n")
print("本次完整结果：", RUN / "result.json")

,检查,数量,结果
0,两版八个 payload 的逻辑内容,8,精确相等
1,复用后 payload/Meta 文件身份,32,保持不变
2,三份隔离输入文件身份,180,保持不变
3,缺少 T+1 分钟的 CLI 负例,1,退出 1；Feature 保留，Label 未发布


本次完整结果： /home/wsw/app/research-evidence/stock-1430-evidence-2026-09-13-a74ry7d6/h03-run-sl1ucp7y/result.json


## 事实的适用范围

本 Notebook 的输出绑定上述 commit、归档、验证脚本和环境。它不把历史运行输出
移植为当前证据；源代码、输入或计算语义变化后，必须按 README 重新判断重验范围。
表格只支持固定样本上的技术复核，不证明历史 14:30 前实际就绪、原始输入从未修订、
全历史 coverage、alpha 或收益；它也不替代 H02 的逐笔守恒证据或 H04 的融合验证。

当前运行目录保存的原始日志包含预设失败。采用评审、状态和结论见 README 的 H03。